# Mixture of Experts

# One Expert vs Multiple Experts

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# One Expert
# -----------------------------
class Expert(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim)
        )

    def forward(self, x):
        return self.net(x)


# -----------------------------
# Mixture of Experts
# -----------------------------
class MoE(nn.Module):
    def __init__(self, hidden_dim=128, num_experts=4):
        super().__init__()

        self.router = nn.Linear(hidden_dim, num_experts)

        self.experts = nn.ModuleList(
            [Expert(hidden_dim) for _ in range(num_experts)]
        )

    def forward(self, x):

        # Router probabilities
        gate = F.softmax(self.router(x), dim=-1)

        # Top-2 experts
        values, indices = torch.topk(gate, k=2, dim=-1)

        output = torch.zeros_like(x)

        # Process selected experts
        for k in range(2):

            idx = indices[..., k]

            for expert_id in range(len(self.experts)):

                mask = idx == expert_id

                if mask.any():

                    expert_out = self.experts[expert_id](x[mask])

                    output[mask] += (
                        values[mask, k].unsqueeze(-1)
                        * expert_out
                    )

        return output


# Example
x = torch.randn(8, 128)

model = MoE()

y = model(x)

print(y.shape)

torch.Size([8, 128])


**Understanding**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# ==========================================================
# Dense Feed Forward
# ==========================================================

class DenseFFN(nn.Module):

    def __init__(self, d_model):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4, d_model)
        )

    def forward(self,x):
        return self.net(x)


# ==========================================================
# One Expert
# ==========================================================

class Expert(nn.Module):

    def __init__(self,d_model):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model,d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4,d_model)
        )

    def forward(self,x):

        return self.net(x)



# ==========================================================
# Mixture Of Experts
# ==========================================================

class SimpleMoE(nn.Module):

    def __init__(self,d_model,num_experts=4):

        super().__init__()

        self.router = nn.Linear(d_model,num_experts)

        self.experts = nn.ModuleList(

            [Expert(d_model) for _ in range(num_experts)]

        )


    def forward(self,x):

        gate_logits = self.router(x)

        gate = F.softmax(gate_logits,dim=-1)

        values,indices = torch.topk(gate,k=2,dim=-1)

        final_output = torch.zeros_like(x)

        print("="*60)
        print("ROUTER DECISION")
        print("="*60)

        for token in range(len(x)):

            print(f"\nToken {token}")

            print("Router Probabilities")

            print(gate[token])

            print("Selected Experts :",indices[token].tolist())

            token_output = 0

            for k in range(2):

                expert_id = indices[token,k].item()

                weight = values[token,k]

                out = self.experts[expert_id](x[token].unsqueeze(0))

                token_output += weight*out

            final_output[token]=token_output

        return final_output


# ==========================================================
# Example
# ==========================================================

d_model = 8

dense = DenseFFN(d_model)

moe = SimpleMoE(d_model,4)

# Four completely different tokens

x = torch.tensor([

[1.,1.,1.,1.,1.,1.,1.,1.],

[8.,8.,8.,8.,8.,8.,8.,8.],

[-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.],

[2.,5.,1.,7.,3.,8.,9.,0.]

])

print("="*60)
print("INPUT")
print("="*60)

print(x)


print("\n\n")
print("="*60)
print("DENSE FFN OUTPUT")
print("="*60)

dense_output = dense(x)

print(dense_output)


print("\n\n")
print("="*60)
print("MIXTURE OF EXPERTS OUTPUT")
print("="*60)

moe_output = moe(x)

print("\nFinal Output")

print(moe_output)

INPUT
tensor([[ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.],
        [ 8.,  8.,  8.,  8.,  8.,  8.,  8.,  8.],
        [-4., -4., -4., -4., -4., -4., -4., -4.],
        [ 2.,  5.,  1.,  7.,  3.,  8.,  9.,  0.]])



DENSE FFN OUTPUT
tensor([[-0.1419, -0.4574,  0.3479,  0.2028, -0.1224, -0.1032, -0.0645,  0.0737],
        [-2.3594, -3.6954,  1.9492,  0.7040, -1.5319,  0.0257, -0.5355, -1.3260],
        [ 0.1220, -0.4151,  0.5741, -0.1609,  0.9233, -0.8219,  0.4045,  0.6266],
        [-1.3618, -2.1282,  1.3009,  0.6420, -0.8086, -0.4708, -0.4613,  0.2362]],
       grad_fn=<AddmmBackward0>)



MIXTURE OF EXPERTS OUTPUT
ROUTER DECISION

Token 0
Router Probabilities
tensor([0.4774, 0.1607, 0.0766, 0.2853], grad_fn=<SelectBackward0>)
Selected Experts : [0, 3]

Token 1
Router Probabilities
tensor([9.7480e-01, 1.2551e-04, 2.7715e-06, 2.5072e-02],
       grad_fn=<SelectBackward0>)
Selected Experts : [0, 3]

Token 2
Router Probabilities
tensor([0.0020, 0.1869, 0.7997, 0.0114], grad_fn=<SelectBackward0

# From Scratch

**Import Libraries & Environment Setup**

In [ ]:
# ==========================================================
# TinyGPT from Scratch with Mixture of Experts (MoE)
# Block 1 : Imports & Environment Setup
# ==========================================================

import os
import math
import time
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

# ----------------------------------------------------------
# Reproducibility
# ----------------------------------------------------------

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Make experiments reproducible
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ----------------------------------------------------------
# Device
# ----------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ----------------------------------------------------------
# Helper Function
# ----------------------------------------------------------

def count_parameters(model):
    """
    Count trainable parameters
    """

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


# ----------------------------------------------------------
# Display Environment
# ----------------------------------------------------------

print("=" * 60)
print("TinyGPT + Mixture of Experts")
print("=" * 60)

print(f"PyTorch Version : {torch.__version__}")
print(f"Device          : {DEVICE}")
print(f"Random Seed     : {SEED}")

if torch.cuda.is_available():

    print(f"GPU Name        : {torch.cuda.get_device_name(0)}")

print("=" * 60)

TinyGPT + Mixture of Experts
PyTorch Version : 2.11.0+cu128
Device          : cuda
Random Seed     : 42
GPU Name        : Tesla T4


**Hyperparameters**

In [ ]:
# ==========================================================
# Block 2 : Hyperparameters (Updated)
# ==========================================================

# ----------------------------------------------------------
# Dataset
# ----------------------------------------------------------

BATCH_SIZE = 32              # Sequences per batch

BLOCK_SIZE = 64              # Context window

# ----------------------------------------------------------
# Model Architecture
# ----------------------------------------------------------

N_EMBED = 128                # Embedding Dimension

N_HEAD = 4                   # Attention Heads

N_LAYER = 4                  # Transformer Blocks

FFN_EXPANSION = 4

DROPOUT = 0.2

# ----------------------------------------------------------
# Training
# ----------------------------------------------------------

LEARNING_RATE = 3e-4

MAX_ITERS = 5000             # Total training iterations

EVAL_INTERVAL = 500          # Validation frequency

EVAL_ITERS = 100             # Validation batches

WEIGHT_DECAY = 1e-2

# ----------------------------------------------------------
# Generation
# ----------------------------------------------------------

MAX_NEW_TOKENS = 500

TEMPERATURE = 0.8

TOP_K = 20

# ----------------------------------------------------------
# Display Configuration
# ----------------------------------------------------------

print("="*60)
print("MODEL CONFIGURATION")
print("="*60)

print(f"Batch Size         : {BATCH_SIZE}")
print(f"Context Length     : {BLOCK_SIZE}")

print()

print(f"Embedding Size     : {N_EMBED}")
print(f"Attention Heads    : {N_HEAD}")
print(f"Transformer Layers : {N_LAYER}")
print(f"FFN Expansion      : {FFN_EXPANSION}")
print(f"Dropout            : {DROPOUT}")

print()

print(f"Learning Rate      : {LEARNING_RATE}")
print(f"Training Iterations: {MAX_ITERS}")
print(f"Eval Interval      : {EVAL_INTERVAL}")
print(f"Eval Batches       : {EVAL_ITERS}")
print(f"Weight Decay       : {WEIGHT_DECAY}")

print()

print(f"Temperature        : {TEMPERATURE}")
print(f"Top-K Sampling     : {TOP_K}")

print("="*60)

MODEL CONFIGURATION
Batch Size         : 32
Context Length     : 64

Embedding Size     : 128
Attention Heads    : 4
Transformer Layers : 4
FFN Expansion      : 4
Dropout            : 0.2

Learning Rate      : 0.0003
Training Iterations: 5000
Eval Interval      : 500
Eval Batches       : 100
Weight Decay       : 0.01

Temperature        : 0.8
Top-K Sampling     : 20


**Download & Prepare Dataset**

In [ ]:
# ==========================================================
# Block 3 : Download & Prepare Dataset
# ==========================================================

import os
import urllib.request

# ----------------------------------------------------------
# Download Tiny Shakespeare Dataset
# ----------------------------------------------------------

DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

DATA_FILE = "tiny_shakespeare.txt"

if not os.path.exists(DATA_FILE):

    print("Downloading Dataset...\n")

    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

    print("Download Complete!")

else:

    print("Dataset already exists.")


# ----------------------------------------------------------
# Read Dataset
# ----------------------------------------------------------

with open(DATA_FILE, "r", encoding="utf-8") as f:

    text = f.read()

print("="*60)
print("DATASET INFORMATION")
print("="*60)

print(f"Total Characters : {len(text):,}")

print()

print("First 500 Characters\n")

print(text[:500])

print("="*60)


# ----------------------------------------------------------
# Build Vocabulary
# ----------------------------------------------------------

chars = sorted(list(set(text)))

vocab_size = len(chars)

print("\nVocabulary Size :", vocab_size)

print()

print("Vocabulary\n")

print(chars)


# ----------------------------------------------------------
# Character → Integer Mapping
# ----------------------------------------------------------

stoi = {

    ch:i

    for i,ch in enumerate(chars)

}


# ----------------------------------------------------------
# Integer → Character Mapping
# ----------------------------------------------------------

itos = {

    i:ch

    for i,ch in enumerate(chars)

}


# ----------------------------------------------------------
# Encoder
# ----------------------------------------------------------

def encode(text):

    return [

        stoi[c]

        for c in text

    ]


# ----------------------------------------------------------
# Decoder
# ----------------------------------------------------------

def decode(tokens):

    return "".join(

        [

            itos[i]

            for i in tokens

        ]

    )


# ----------------------------------------------------------
# Encode Whole Dataset
# ----------------------------------------------------------

data = torch.tensor(

    encode(text),

    dtype=torch.long

)


print("\n")

print("="*60)
print("ENCODING DEMO")
print("="*60)

sample = "Hello GPT"

encoded = encode(sample)

decoded = decode(encoded)

print("Original Text")

print(sample)

print()

print("Encoded")

print(encoded)

print()

print("Decoded")

print(decoded)

print()

print("="*60)
print("DATA TENSOR")
print("="*60)

print(data)

print()

print("Shape :", data.shape)


Download Complete!
DATASET INFORMATION
Total Characters : 1,115,394

First 500 Characters

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor

Vocabulary Size : 65

Vocabulary

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


ENCODING DEMO
Original Text
Hello GPT

Enco

**Train / Validation Split & Batch Generator**

In [ ]:
# ==========================================================
# Block 4 : Train / Validation Split & Batch Generator
# ==========================================================

# ----------------------------------------------------------
# Train / Validation Split
# ----------------------------------------------------------

split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

print("=" * 60)
print("DATA SPLIT")
print("=" * 60)

print(f"Training Characters   : {len(train_data):,}")
print(f"Validation Characters : {len(val_data):,}")

print()


# ----------------------------------------------------------
# Batch Generator
# ----------------------------------------------------------

def get_batch(split="train"):

    dataset = train_data if split == "train" else val_data

    # Random starting positions
    ix = torch.randint(
        len(dataset) - BLOCK_SIZE,
        (BATCH_SIZE,)
    )

    # Input
    x = torch.stack([
        dataset[i:i+BLOCK_SIZE]
        for i in ix
    ])

    # Target (Shifted by 1)
    y = torch.stack([
        dataset[i+1:i+BLOCK_SIZE+1]
        for i in ix
    ])

    return x.to(DEVICE), y.to(DEVICE)


# ----------------------------------------------------------
# Generate One Batch
# ----------------------------------------------------------

x, y = get_batch("train")


print("=" * 60)
print("BATCH INFORMATION")
print("=" * 60)

print("Input Shape  :", x.shape)
print("Target Shape :", y.shape)

print()


# ----------------------------------------------------------
# Show One Training Example
# ----------------------------------------------------------

print("=" * 60)
print("NEXT TOKEN PREDICTION DEMO")
print("=" * 60)

sample_x = x[0].cpu().tolist()
sample_y = y[0].cpu().tolist()

print()

print("Input IDs")

print(sample_x)

print()

print("Target IDs")

print(sample_y)

print()

print("Input Text")

print(decode(sample_x))

print()

print("Target Text")

print(decode(sample_y))

print("=" * 60)

DATA SPLIT
Training Characters   : 1,003,854
Validation Characters : 111,540

BATCH INFORMATION
Input Shape  : torch.Size([32, 64])
Target Shape : torch.Size([32, 64])

NEXT TOKEN PREDICTION DEMO

Input IDs
[28, 39, 56, 47, 57, 2, 0, 35, 46, 39, 58, 1, 57, 39, 47, 42, 1, 51, 63, 1, 51, 39, 52, 6, 1, 61, 46, 43, 52, 1, 51, 63, 1, 40, 43, 58, 53, 57, 57, 43, 42, 1, 57, 53, 59, 50, 0, 16, 47, 42, 1, 52, 53, 58, 1, 39, 58, 58, 43, 52, 42, 1, 46, 47]

Target IDs
[39, 56, 47, 57, 2, 0, 35, 46, 39, 58, 1, 57, 39, 47, 42, 1, 51, 63, 1, 51, 39, 52, 6, 1, 61, 46, 43, 52, 1, 51, 63, 1, 40, 43, 58, 53, 57, 57, 43, 42, 1, 57, 53, 59, 50, 0, 16, 47, 42, 1, 52, 53, 58, 1, 39, 58, 58, 43, 52, 42, 1, 46, 47, 51]

Input Text
Paris!
What said my man, when my betossed soul
Did not attend hi

Target Text
aris!
What said my man, when my betossed soul
Did not attend him


**Token & Position Embedding**

In [ ]:
# ==========================================================
# Block 5 : Token & Position Embedding
# ==========================================================

class InputEmbedding(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------------------------
        # Token Embedding
        # ----------------------------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            N_EMBED
        )

        # ----------------------------------------------
        # Position Embedding
        # ----------------------------------------------

        self.position_embedding = nn.Embedding(
            BLOCK_SIZE,
            N_EMBED
        )

    def forward(self, x):

        B, T = x.shape

        # ----------------------------------------------
        # Token Embedding
        # Shape:
        # (B,T) → (B,T,N_EMBED)
        # ----------------------------------------------

        token = self.token_embedding(x)

        # ----------------------------------------------
        # Position IDs
        # ----------------------------------------------

        position_ids = torch.arange(
            T,
            device=x.device
        )

        # ----------------------------------------------
        # Position Embedding
        # Shape:
        # (T) → (T,N_EMBED)
        # ----------------------------------------------

        position = self.position_embedding(position_ids)

        # ----------------------------------------------
        # Final Embedding
        # ----------------------------------------------

        embedding = token + position

        return embedding

**Test Embedding Layer**

In [ ]:
# ==========================================================
# Test Embedding Layer
# ==========================================================

embedding_layer = InputEmbedding().to(DEVICE)

x, y = get_batch()

out = embedding_layer(x)

print("="*60)
print("EMBEDDING OUTPUT")
print("="*60)

print("Input Shape")

print(x.shape)

print()

print("Embedding Shape")

print(out.shape)

print()

print("One Token Embedding")

print(out[0,0])

print()

print("Embedding Dimension")

print(out[0,0].shape)

print()

print("Total Parameters")

print(count_parameters(embedding_layer))

EMBEDDING OUTPUT
Input Shape
torch.Size([32, 64])

Embedding Shape
torch.Size([32, 64, 128])

One Token Embedding
tensor([ 0.0169,  1.0302,  0.3572,  3.1215,  0.0941, -0.5223,  0.0390, -0.7644,
         0.0501, -0.9882,  0.1576,  0.6351, -1.7535,  0.1148, -1.5976, -0.0542,
        -0.2566, -1.6563, -0.2635,  0.5058,  1.6332,  0.6918,  0.7925,  3.1038,
        -0.7558, -1.5187,  1.4885, -0.0708, -0.9361, -1.8773,  0.4821,  0.0195,
        -3.8635, -2.2473,  0.2041,  1.0502, -0.3502,  3.6899,  2.2061,  1.2469,
        -1.9195, -0.0793, -4.1497, -0.4702,  2.7407, -0.7074,  0.6855, -3.0850,
        -1.1793, -0.0070, -1.1980, -1.3035,  1.2982, -0.3769, -2.4343, -0.4401,
        -2.8647,  0.7879,  1.0080,  2.8744,  1.5277, -0.4302, -2.9297,  0.3544,
        -2.2976,  2.8467,  1.0878, -2.1000,  0.9087, -2.1927,  0.6724, -0.3744,
         0.2635,  0.5547, -0.1762, -0.4188, -0.8761, -2.0676,  0.3089, -1.5396,
         0.1013,  3.5130, -0.2105, -0.7694,  0.3100, -0.5348, -4.0915,  0.0632,
      

**Single Self Attention Head**

In [ ]:
# ==========================================================
# Block 6A : Single Self Attention Head
# ==========================================================

class SelfAttentionHead(nn.Module):

    def __init__(self):

        super().__init__()

        # Head Dimension
        self.head_dim = N_EMBED // N_HEAD

        # Q Projection
        self.query = nn.Linear(
            N_EMBED,
            self.head_dim,
            bias=False
        )

        # K Projection
        self.key = nn.Linear(
            N_EMBED,
            self.head_dim,
            bias=False
        )

        # V Projection
        self.value = nn.Linear(
            N_EMBED,
            self.head_dim,
            bias=False
        )


    def forward(self, x):

        B, T, C = x.shape

        # ----------------------------------------
        # Query
        # ----------------------------------------

        Q = self.query(x)

        # ----------------------------------------
        # Key
        # ----------------------------------------

        K = self.key(x)

        # ----------------------------------------
        # Value
        # ----------------------------------------

        V = self.value(x)

        # ----------------------------------------
        # Attention Scores
        # ----------------------------------------

        scores = Q @ K.transpose(-2, -1)

        # Scaling
        scores = scores / math.sqrt(self.head_dim)

        # Softmax
        weights = F.softmax(scores, dim=-1)

        # Output
        out = weights @ V

        return out, Q, K, V, scores, weights

**Test Self Attention Head**

In [ ]:
# ==========================================================
# Test Self Attention Head
# ==========================================================

embedding_layer = InputEmbedding().to(DEVICE)

attention = SelfAttentionHead().to(DEVICE)

x, _ = get_batch()

embeddings = embedding_layer(x)

out, Q, K, V, scores, weights = attention(embeddings)

print("="*60)
print("SELF ATTENTION")
print("="*60)

print()

print("Input Shape")
print(embeddings.shape)

print()

print("Query Shape")
print(Q.shape)

print()

print("Key Shape")
print(K.shape)

print()

print("Value Shape")
print(V.shape)

print()

print("Attention Score Shape")
print(scores.shape)

print()

print("Attention Weight Shape")
print(weights.shape)

print()

print("Output Shape")
print(out.shape)

print()

print("="*60)

print("Attention Matrix (First 5 x 5)")

print(weights[0,:5,:5])

SELF ATTENTION

Input Shape
torch.Size([32, 64, 128])

Query Shape
torch.Size([32, 64, 32])

Key Shape
torch.Size([32, 64, 32])

Value Shape
torch.Size([32, 64, 32])

Attention Score Shape
torch.Size([32, 64, 64])

Attention Weight Shape
torch.Size([32, 64, 64])

Output Shape
torch.Size([32, 64, 32])

Attention Matrix (First 5 x 5)
tensor([[0.0526, 0.0045, 0.0071, 0.0048, 0.0139],
        [0.0105, 0.0239, 0.0136, 0.0150, 0.0066],
        [0.0020, 0.0172, 0.0174, 0.0110, 0.0041],
        [0.0214, 0.0081, 0.0174, 0.0099, 0.0126],
        [0.0060, 0.0231, 0.0090, 0.0078, 0.0105]], device='cuda:0',
       grad_fn=<SliceBackward0>)


**Causal Masked Self Attention**

In [ ]:
# ==========================================================
# Block 6B : Causal Masked Self Attention
# ==========================================================

class SelfAttentionHead(nn.Module):

    def __init__(self):

        super().__init__()

        self.head_dim = N_EMBED // N_HEAD

        self.query = nn.Linear(
            N_EMBED,
            self.head_dim,
            bias=False
        )

        self.key = nn.Linear(
            N_EMBED,
            self.head_dim,
            bias=False
        )

        self.value = nn.Linear(
            N_EMBED,
            self.head_dim,
            bias=False
        )

        # Lower triangular matrix
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE))
        )


    def forward(self, x):

        B, T, C = x.shape

        Q = self.query(x)

        K = self.key(x)

        V = self.value(x)

        # -----------------------------------------
        # Attention Scores
        # -----------------------------------------

        scores = Q @ K.transpose(-2, -1)

        scores = scores / math.sqrt(self.head_dim)

        # -----------------------------------------
        # Causal Mask
        # -----------------------------------------

        scores = scores.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        # -----------------------------------------
        # Softmax
        # -----------------------------------------

        weights = F.softmax(scores, dim=-1)

        # -----------------------------------------
        # Output
        # -----------------------------------------

        out = weights @ V

        return out, weights

** Test Causal Masked Self Attention**

In [ ]:
# ==========================================================
# Test Causal Attention
# ==========================================================

embedding_layer = InputEmbedding().to(DEVICE)

attention = SelfAttentionHead().to(DEVICE)

x, _ = get_batch()

embeddings = embedding_layer(x)

out, weights = attention(embeddings)

print("="*60)
print("MASKED SELF ATTENTION")
print("="*60)

print()

print("Attention Weight Shape")

print(weights.shape)

print()

print("First 8 x 8 Attention Matrix")

print(weights[0,:8,:8])

MASKED SELF ATTENTION

Attention Weight Shape
torch.Size([32, 64, 64])

First 8 x 8 Attention Matrix
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3463, 0.6537, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5379, 0.1449, 0.3172, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0934, 0.3423, 0.1712, 0.3931, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0273, 0.1309, 0.0647, 0.0319, 0.7452, 0.0000, 0.0000, 0.0000],
        [0.1849, 0.0786, 0.0915, 0.0309, 0.2931, 0.3211, 0.0000, 0.0000],
        [0.0384, 0.2285, 0.1017, 0.0804, 0.3026, 0.0523, 0.1962, 0.0000],
        [0.2983, 0.0486, 0.0579, 0.0909, 0.1637, 0.1619, 0.0712, 0.1075]],
       device='cuda:0', grad_fn=<SliceBackward0>)


**Multi Head Self Attention**

In [ ]:
# ==========================================================
# Block 6C : Multi Head Self Attention
# ==========================================================

class MultiHeadAttention(nn.Module):

    def __init__(self):

        super().__init__()

        self.heads = nn.ModuleList([
            SelfAttentionHead()
            for _ in range(N_HEAD)
        ])

        self.projection = nn.Linear(
            N_EMBED,
            N_EMBED
        )

        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):

        head_outputs = []

        for head in self.heads:

            out, _ = head(x)

            head_outputs.append(out)

        out = torch.cat(head_outputs, dim=-1)

        out = self.projection(out)

        out = self.dropout(out)

        return out

**Test Multi Head Attention**

In [ ]:
# ==========================================================
# Test Multi Head Attention
# ==========================================================

embedding_layer = InputEmbedding().to(DEVICE)

multi_head = MultiHeadAttention().to(DEVICE)

x, _ = get_batch()

embeddings = embedding_layer(x)

output = multi_head(embeddings)

print("="*60)
print("MULTI HEAD ATTENTION")
print("="*60)

print()

print("Input Shape")
print(embeddings.shape)

print()

print("Output Shape")
print(output.shape)

print()

print("Number of Heads")
print(N_HEAD)

print()

print("Head Dimension")
print(N_EMBED // N_HEAD)

print()

print("Total Parameters")
print(f"{count_parameters(multi_head):,}")

MULTI HEAD ATTENTION

Input Shape
torch.Size([32, 64, 128])

Output Shape
torch.Size([32, 64, 128])

Number of Heads
4

Head Dimension
32

Total Parameters
65,664


**FFN**

In [ ]:
# ==========================================================
# Block 7 : Feed Forward Network (Dense)
# ==========================================================

class FeedForward(nn.Module):

    def __init__(self):

        super().__init__()

        hidden_dim = N_EMBED * FFN_EXPANSION

        self.net = nn.Sequential(

            nn.Linear(
                N_EMBED,
                hidden_dim
            ),

            nn.GELU(),

            nn.Linear(
                hidden_dim,
                N_EMBED
            ),

            nn.Dropout(DROPOUT)

        )

    def forward(self, x):

        return self.net(x)

**Test FFN**

In [ ]:
# ==========================================================
# Test Feed Forward Network
# ==========================================================

embedding_layer = InputEmbedding().to(DEVICE)

ffn = FeedForward().to(DEVICE)

x, _ = get_batch()

embeddings = embedding_layer(x)

output = ffn(embeddings)

print("="*60)
print("FEED FORWARD NETWORK")
print("="*60)

print()

print("Input Shape")

print(embeddings.shape)

print()

print("Output Shape")

print(output.shape)

print()

print("First Token Output")

print(output[0,0])

print()

print("Output Dimension")

print(output[0,0].shape)

print()

print("Trainable Parameters")

print(count_parameters(ffn))

FEED FORWARD NETWORK

Input Shape
torch.Size([32, 64, 128])

Output Shape
torch.Size([32, 64, 128])

First Token Output
tensor([-0.4489, -0.0000, -0.1453,  0.2998,  0.0000,  0.1181, -0.1161, -0.4499,
         0.1411,  0.0000,  0.3292, -0.1624,  0.2221,  0.1306, -0.1319, -0.4419,
        -0.2429, -0.0000, -0.1973,  0.2071,  0.1417, -0.2624, -0.3253, -0.2684,
        -0.1838,  0.3872,  0.2752, -0.4522, -0.0000,  0.0723,  0.0944,  0.0000,
         0.2663,  0.0371,  0.3040, -0.4120,  0.4654, -0.3541, -0.0000,  0.2363,
        -0.1639,  0.1127, -0.1355, -0.3923, -0.0725, -0.0000, -0.0772, -0.2992,
         0.0000, -0.1952,  0.2537, -0.0070, -0.2556, -0.5373, -0.0284, -0.2489,
        -0.1419,  0.0000,  0.4953,  0.3918,  0.2872, -0.0423, -0.0301, -0.5142,
        -0.1933,  0.2482,  0.1415,  0.1477, -0.6454,  0.0646,  0.0739, -0.2968,
        -0.3992,  0.5644,  0.0000,  0.2101,  0.0000, -0.0219, -0.0000,  0.0449,
        -0.2667,  0.0000, -0.0242, -0.4464, -0.1270, -0.1701,  0.2962,  0.0455,


**Layer Normalization**

In [ ]:
# ==========================================================
# Block 8A : Layer Normalization (Scratch)
# ==========================================================

class LayerNorm(nn.Module):

    def __init__(self, dim, eps=1e-5):

        super().__init__()

        self.eps = eps

        self.gamma = nn.Parameter(torch.ones(dim))

        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):

        mean = x.mean(dim=-1, keepdim=True)

        variance = ((x - mean) ** 2).mean(dim=-1, keepdim=True)

        std = torch.sqrt(variance + self.eps)

        x = (x - mean) / std

        return self.gamma * x + self.beta

**Output**

In [ ]:
layer_norm = LayerNorm(N_EMBED).to(DEVICE)

x,_ = get_batch()

embedding = InputEmbedding().to(DEVICE)

emb = embedding(x)

output = layer_norm(emb)

print("="*60)
print("LAYER NORMALIZATION")
print("="*60)

print("Input Shape :", emb.shape)

print("Output Shape:", output.shape)

print()

print("Mean of First Token")

print(output[0,0].mean())

print()

print("Std of First Token")

print(output[0,0].std())

LAYER NORMALIZATION
Input Shape : torch.Size([32, 64, 128])
Output Shape: torch.Size([32, 64, 128])

Mean of First Token
tensor(-1.4901e-08, device='cuda:0', grad_fn=<MeanBackward0>)

Std of First Token
tensor(1.0039, device='cuda:0', grad_fn=<StdBackward0>)


**Transformer Block**

In [ ]:
# ==========================================================
# Block 8B : Transformer Block
# ==========================================================

class TransformerBlock(nn.Module):

    def __init__(self):

        super().__init__()

        self.ln1 = LayerNorm(N_EMBED)

        self.attention = MultiHeadAttention()

        self.ln2 = LayerNorm(N_EMBED)

        self.ffn = FeedForward()

    def forward(self, x):

        x = x + self.attention(self.ln1(x))

        x = x + self.ffn(self.ln2(x))

        return x

**Testing**

In [ ]:
# ==========================================================
# Test Transformer Block
# ==========================================================

embedding_layer = InputEmbedding().to(DEVICE)

transformer = TransformerBlock().to(DEVICE)

x, _ = get_batch()

emb = embedding_layer(x)

output = transformer(emb)

print("=" * 60)
print("TRANSFORMER BLOCK")
print("=" * 60)

print()

print("Input Shape")
print(emb.shape)

print()

print("Output Shape")
print(output.shape)

print()

print("Attention Heads")
print(N_HEAD)

print()

print("Embedding Dimension")
print(N_EMBED)

print()

print("Total Parameters")
print(f"{count_parameters(transformer):,}")

TRANSFORMER BLOCK

Input Shape
torch.Size([32, 64, 128])

Output Shape
torch.Size([32, 64, 128])

Attention Heads
4

Embedding Dimension
128

Total Parameters
197,888


**GPT IDEA**

In [ ]:
# ==========================================================
# Block 9 : TinyGPT
# ==========================================================

class TinyGPT(nn.Module):

    def __init__(self):

        super().__init__()

        # --------------------------------------------------
        # Input Embedding
        # --------------------------------------------------

        self.embedding = InputEmbedding()

        # --------------------------------------------------
        # Transformer Blocks
        # --------------------------------------------------

        self.blocks = nn.Sequential(
            *[TransformerBlock() for _ in range(N_LAYER)]
        )

        # --------------------------------------------------
        # Final LayerNorm
        # --------------------------------------------------

        self.ln_final = LayerNorm(N_EMBED)

        # --------------------------------------------------
        # Language Modeling Head
        # --------------------------------------------------

        self.lm_head = nn.Linear(
            N_EMBED,
            vocab_size
        )

    # ======================================================
    # Forward Pass
    # ======================================================

    def forward(self, idx, targets=None):

        # Embedding
        x = self.embedding(idx)

        # Transformer Blocks
        x = self.blocks(x)

        # Final LayerNorm
        x = self.ln_final(x)

        # Vocabulary Logits
        logits = self.lm_head(x)

        loss = None

        # --------------------------------------------------
        # Training Mode
        # --------------------------------------------------

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.reshape(B * T, C)

            targets = targets.reshape(B * T)

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

    # ======================================================
    # Text Generation
    # ======================================================

    @torch.no_grad()
    def generate(
        self,
        idx,
        max_new_tokens,
        temperature=1.0,
        top_k=None
    ):

        self.eval()

        for _ in range(max_new_tokens):

            # ------------------------------------------
            # Keep only last BLOCK_SIZE tokens
            # ------------------------------------------

            idx_cond = idx[:, -BLOCK_SIZE:]

            # ------------------------------------------
            # Forward Pass
            # ------------------------------------------

            logits, _ = self(idx_cond)

            # Last Token Only
            logits = logits[:, -1, :]

            # ------------------------------------------
            # Temperature
            # ------------------------------------------

            logits = logits / temperature

            # ------------------------------------------
            # Top-K Sampling
            # ------------------------------------------

            if top_k is not None:

                values, indices = torch.topk(logits, top_k)

                probs = F.softmax(values, dim=-1)

                next_local = torch.multinomial(
                    probs,
                    num_samples=1
                )

                next_token = indices.gather(
                    -1,
                    next_local
                )

            else:

                probs = F.softmax(logits, dim=-1)

                next_token = torch.multinomial(
                    probs,
                    num_samples=1
                )

            # ------------------------------------------
            # Append Prediction
            # ------------------------------------------

            idx = torch.cat(
                (idx, next_token),
                dim=1
            )

        self.train()

        return idx

**Test**

In [ ]:
# ==========================================================
# Test TinyGPT
# ==========================================================

model = TinyGPT().to(DEVICE)

print("="*60)
print("TinyGPT SUMMARY")
print("="*60)

print()

print(model)

print()

print("Total Parameters")

print(f"{count_parameters(model):,}")

print()

x,y = get_batch()

logits,loss = model(x,y)

print()

print("Logits Shape")

print(logits.shape)

print()

print("Loss")

print(loss.item())

TinyGPT SUMMARY

TinyGPT(
  (embedding): InputEmbedding(
    (token_embedding): Embedding(65, 128)
    (position_embedding): Embedding(64, 128)
  )
  (blocks): Sequential(
    (0): TransformerBlock(
      (ln1): LayerNorm()
      (attention): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x SelfAttentionHead(
            (query): Linear(in_features=128, out_features=32, bias=False)
            (key): Linear(in_features=128, out_features=32, bias=False)
            (value): Linear(in_features=128, out_features=32, bias=False)
          )
        )
        (projection): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ln2): LayerNorm()
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=128, bias=True)
          (3): Dropout(p=0.2, inplace=Fa

**Create Model**

In [ ]:
# ==========================================================
# Block 10A : Create Model
# ==========================================================

model = TinyGPT().to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("="*60)
print("TRAINING SETUP")
print("="*60)

print()

print("Total Parameters")

print(f"{count_parameters(model):,}")

print()

print("Device")

print(DEVICE)

TRAINING SETUP

Total Parameters
816,705

Device
cuda


**Loss**

In [ ]:
# ==========================================================
# Block 10B : Estimate Loss
# ==========================================================

@torch.no_grad()
def estimate_loss():

    model.eval()

    losses = {}

    for split in ["train", "val"]:

        split_losses = torch.zeros(EVAL_ITERS)

        for k in range(EVAL_ITERS):

            x, y = get_batch(split)

            _, loss = model(x, y)

            split_losses[k] = loss.item()

        losses[split] = split_losses.mean()

    model.train()

    return losses

**Loop**

In [ ]:
# ==========================================================
# Block 10C : Training Loop
# ==========================================================

print("="*60)
print("START TRAINING")
print("="*60)

loss_history = []
val_history = []

for step in range(MAX_ITERS):

    if step % EVAL_INTERVAL == 0:

        losses = estimate_loss()

        train_loss = losses["train"].item()
        val_loss = losses["val"].item()

        loss_history.append(train_loss)
        val_history.append(val_loss)

        print(
            f"Step {step:4d} | "
            f"Train Loss = {train_loss:.4f} | "
            f"Val Loss = {val_loss:.4f}"
        )

    x, y = get_batch("train")

    logits, loss = model(x, y)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

print()

print("="*60)
print("TRAINING FINISHED")
print("="*60)

START TRAINING
Step    0 | Train Loss = 4.3418 | Val Loss = 4.3417
Step  500 | Train Loss = 2.3020 | Val Loss = 2.3138
Step 1000 | Train Loss = 2.0646 | Val Loss = 2.1070
Step 1500 | Train Loss = 1.9178 | Val Loss = 1.9977
Step 2000 | Train Loss = 1.8143 | Val Loss = 1.9233
Step 2500 | Train Loss = 1.7275 | Val Loss = 1.8704
Step 3000 | Train Loss = 1.6839 | Val Loss = 1.8340
Step 3500 | Train Loss = 1.6449 | Val Loss = 1.8072
Step 4000 | Train Loss = 1.6032 | Val Loss = 1.7788
Step 4500 | Train Loss = 1.5782 | Val Loss = 1.7494

TRAINING FINISHED


**GENERATION**

In [ ]:
# ==========================================================
# Block 11A : Text Generation Method
# ==========================================================

@torch.no_grad()
def generate(self, idx, max_new_tokens):

    self.eval()

    for _ in range(max_new_tokens):

        # Keep only last BLOCK_SIZE tokens
        idx_cond = idx[:, -BLOCK_SIZE:]

        # Forward pass
        logits, _ = self(idx_cond)

        # Last token prediction
        logits = logits[:, -1, :]

        # Convert logits → probabilities
        probs = F.softmax(logits, dim=-1)

        # Sample next character
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        # Append prediction
        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    return idx

**Test**

In [ ]:
# ==========================================================
# Block 11B : Generate Shakespeare Text
# ==========================================================

context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=DEVICE
)

generated = model.generate(
    context,
    max_new_tokens=500
)

text = decode(
    generated[0].tolist()
)

print("=" * 60)
print("GENERATED TEXT")
print("=" * 60)
print()
print(text)

GENERATED TEXT


LADY VINDIO:
And wish you come himfict ty mal.

WARWICK:
Then to with him?---God we and rouse you fack,
Sir?

Mordan:
All if it till the prover unjest, my gover.

ROMEO:
A lears this cage the littles is to the call
A better of sume of your hasten son most-shick up?
Well may long? Nor noble sheed, safe, that the Xow
Prosed to gue of the desaines; our keep,
And In then would you bege, these is remitters ves
With before, but to joy my dught and in their to at
Besalitence. Yorks.
For and allial like


**NOW MoE's**

In [ ]:
# ==========================================================
# Block 12A : Expert Network
# ==========================================================

class Expert(nn.Module):

    def __init__(self):

        super().__init__()

        hidden_dim = N_EMBED * FFN_EXPANSION

        self.network = nn.Sequential(

            nn.Linear(
                N_EMBED,
                hidden_dim
            ),

            nn.GELU(),

            nn.Linear(
                hidden_dim,
                N_EMBED
            ),

            nn.Dropout(DROPOUT)

        )

    def forward(self, x):

        return self.network(x)

**Test**

In [ ]:
# ==========================================================
# Test Expert Network
# ==========================================================

expert = Expert().to(DEVICE)

embedding = InputEmbedding().to(DEVICE)

x, _ = get_batch()

emb = embedding(x)

output = expert(emb)

print("="*60)
print("EXPERT NETWORK")
print("="*60)

print()

print("Input Shape")
print(emb.shape)

print()

print("Output Shape")
print(output.shape)

print()

print("Trainable Parameters")
print(f"{count_parameters(expert):,}")

EXPERT NETWORK

Input Shape
torch.Size([32, 64, 128])

Output Shape
torch.Size([32, 64, 128])

Trainable Parameters
131,712


**Gating Router**

In [ ]:
# ==========================================================
# Block 12B : Router (Gating Network)
# ==========================================================

NUM_EXPERTS = 4

class Router(nn.Module):

    def __init__(self):

        super().__init__()

        self.gate = nn.Linear(
            N_EMBED,
            NUM_EXPERTS
        )

    def forward(self, x):

        # x : (B,T,N_EMBED)

        router_logits = self.gate(x)

        return router_logits

**Test**

In [ ]:
# ==========================================================
# Test Router
# ==========================================================

router = Router().to(DEVICE)

embedding = InputEmbedding().to(DEVICE)

x, _ = get_batch()

emb = embedding(x)

router_logits = router(emb)

print("="*60)
print("ROUTER")
print("="*60)

print()

print("Input Shape")
print(emb.shape)

print()

print("Router Logits Shape")
print(router_logits.shape)

print()

print("First Token Scores")

print(router_logits[0,0])

print()

print("Trainable Parameters")

print(f"{count_parameters(router):,}")

ROUTER

Input Shape
torch.Size([32, 64, 128])

Router Logits Shape
torch.Size([32, 64, 4])

First Token Scores
tensor([ 0.7801,  0.0498, -0.2887,  0.3374], device='cuda:0',
       grad_fn=<SelectBackward0>)

Trainable Parameters
516


**Choose**

In [ ]:
# ==========================================================
# Block 12C : Top-K Routing
# ==========================================================

TOP_K = 2

class TopKRouter(nn.Module):

    def __init__(self):

        super().__init__()

        self.router = Router()

    def forward(self, x):

        # ------------------------------------------
        # Router Scores
        # Shape : (B,T,NUM_EXPERTS)
        # ------------------------------------------

        router_logits = self.router(x)

        # ------------------------------------------
        # Convert to Probabilities
        # ------------------------------------------

        router_probs = torch.softmax(
            router_logits,
            dim=-1
        )

        # ------------------------------------------
        # Select Top-K Experts
        # ------------------------------------------

        topk_probs, topk_indices = torch.topk(
            router_probs,
            k=TOP_K,
            dim=-1
        )

        return (
            router_logits,
            router_probs,
            topk_probs,
            topk_indices
        )

**Test of Choice**

In [ ]:
# ==========================================================
# Test Top-K Router
# ==========================================================

router = TopKRouter().to(DEVICE)

embedding = InputEmbedding().to(DEVICE)

x, _ = get_batch()

emb = embedding(x)

(
    logits,
    probs,
    topk_probs,
    topk_indices
) = router(emb)

print("="*60)
print("TOP-K ROUTER")
print("="*60)

print()

print("Logits Shape")
print(logits.shape)

print()

print("Probability Shape")
print(probs.shape)

print()

print("Top-K Probability Shape")
print(topk_probs.shape)

print()

print("Top-K Index Shape")
print(topk_indices.shape)

print()

print("First Token Probabilities")

print(probs[0,0])

print()

print("Selected Experts")

print(topk_indices[0,0])

print()

print("Selected Probabilities")

print(topk_probs[0,0])

print()

print("Probability Sum")

print(probs[0,0].sum())

TOP-K ROUTER

Logits Shape
torch.Size([32, 64, 4])

Probability Shape
torch.Size([32, 64, 4])

Top-K Probability Shape
torch.Size([32, 64, 2])

Top-K Index Shape
torch.Size([32, 64, 2])

First Token Probabilities
tensor([0.0570, 0.7047, 0.0343, 0.2041], device='cuda:0',
       grad_fn=<SelectBackward0>)

Selected Experts
tensor([1, 3], device='cuda:0')

Selected Probabilities
tensor([0.7047, 0.2041], device='cuda:0', grad_fn=<SelectBackward0>)

Probability Sum
tensor(1., device='cuda:0', grad_fn=<SumBackward0>)


**Now Mixture of Experts**

In [ ]:
# ==========================================================
# Block 12D : Mixture of Experts Layer
# ==========================================================

NUM_EXPERTS = 4
TOP_K = 2


class MoELayer(nn.Module):

    def __init__(self):

        super().__init__()

        # ------------------------------------------
        # Router
        # ------------------------------------------

        self.router = TopKRouter()

        # ------------------------------------------
        # Experts
        # ------------------------------------------

        self.experts = nn.ModuleList([
            Expert()
            for _ in range(NUM_EXPERTS)
        ])

    def forward(self, x):

        (
            router_logits,
            router_probs,
            topk_probs,
            topk_indices
        ) = self.router(x)

        # ------------------------------------------
        # Output Tensor
        # ------------------------------------------

        output = torch.zeros_like(x)

        # ------------------------------------------
        # Run Selected Experts
        # ------------------------------------------

        for k in range(TOP_K):

            expert_index = topk_indices[..., k]

            expert_weight = topk_probs[..., k].unsqueeze(-1)

            expert_output = torch.zeros_like(x)

            for expert_id in range(NUM_EXPERTS):

                mask = (expert_index == expert_id)

                if mask.any():

                    current = self.experts[expert_id](x)

                    expert_output += (
                        current *
                        mask.unsqueeze(-1)
                    )

            output += expert_output * expert_weight

        return output

**Testing**

In [ ]:
# ==========================================================
# Test MoE Layer
# ==========================================================

embedding = InputEmbedding().to(DEVICE)

moe = MoELayer().to(DEVICE)

x, _ = get_batch()

emb = embedding(x)

output = moe(emb)

print("="*60)
print("MIXTURE OF EXPERTS")
print("="*60)

print()

print("Input Shape")
print(emb.shape)

print()

print("Output Shape")
print(output.shape)

print()

print("Experts")
print(NUM_EXPERTS)

print()

print("Top-K")
print(TOP_K)

print()

print("Parameters")
print(f"{count_parameters(moe):,}")

MIXTURE OF EXPERTS

Input Shape
torch.Size([32, 64, 128])

Output Shape
torch.Size([32, 64, 128])

Experts
4

Top-K
2

Parameters
527,364


**Transformer Block**

In [ ]:
# ==========================================================
# Block 12E : Transformer Block with MoE
# ==========================================================

class MoETransformerBlock(nn.Module):

    def __init__(self):

        super().__init__()

        self.ln1 = LayerNorm(N_EMBED)

        self.attention = MultiHeadAttention()

        self.ln2 = LayerNorm(N_EMBED)

        self.moe = MoELayer()

    def forward(self, x):

        # -----------------------------
        # Multi-Head Self Attention
        # -----------------------------

        x = x + self.attention(self.ln1(x))

        # -----------------------------
        # Mixture of Experts
        # -----------------------------

        x = x + self.moe(self.ln2(x))

        return x

**Testing**

In [ ]:
# ==========================================================
# Test MoE Transformer Block
# ==========================================================

embedding = InputEmbedding().to(DEVICE)

block = MoETransformerBlock().to(DEVICE)

x, _ = get_batch()

emb = embedding(x)

output = block(emb)

print("=" * 60)
print("MoE TRANSFORMER BLOCK")
print("=" * 60)

print()

print("Input Shape")
print(emb.shape)

print()

print("Output Shape")
print(output.shape)

print()

print("Attention Heads")
print(N_HEAD)

print()

print("Experts")
print(NUM_EXPERTS)

print()

print("Top-K")
print(TOP_K)

print()

print("Trainable Parameters")
print(f"{count_parameters(block):,}")

MoE TRANSFORMER BLOCK

Input Shape
torch.Size([32, 64, 128])

Output Shape
torch.Size([32, 64, 128])

Attention Heads
4

Experts
4

Top-K
2

Trainable Parameters
593,540


**Tiny GPT MODELIDEA**

In [ ]:
# ==========================================================
# Block 14A : TinyGPT-MoE
# ==========================================================

class TinyGPTMoE(nn.Module):

    def __init__(self):

        super().__init__()

        # ------------------------------------------
        # Embeddings
        # ------------------------------------------

        self.embedding = InputEmbedding()

        # ------------------------------------------
        # MoE Transformer Blocks
        # ------------------------------------------

        self.blocks = nn.Sequential(

            *[
                MoETransformerBlock()
                for _ in range(N_LAYER)
            ]

        )

        # ------------------------------------------
        # Final LayerNorm
        # ------------------------------------------

        self.ln_final = LayerNorm(N_EMBED)

        # ------------------------------------------
        # Language Modeling Head
        # ------------------------------------------

        self.lm_head = nn.Linear(
            N_EMBED,
            vocab_size
        )

    def forward(self, idx, targets=None):

        x = self.embedding(idx)

        x = self.blocks(x)

        x = self.ln_final(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)

            targets = targets.view(B * T)

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx,
        max_new_tokens,
        temperature=1.0,
        top_k=None
    ):

        self.eval()

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -BLOCK_SIZE:]

            logits, _ = self(idx_cond)

            logits = logits[:, -1, :]

            logits = logits / temperature

            if top_k is not None:

                v, _ = torch.topk(
                    logits,
                    top_k
                )

                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(
                logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

            idx = torch.cat(
                (idx, next_token),
                dim=1
            )

        self.train()

        return idx

**Training**

In [ ]:
# ==========================================================
# Train TinyGPT-MoE
# ==========================================================

moe_model = TinyGPTMoE().to(DEVICE)

print("=" * 60)
print("TinyGPT-MoE")
print("=" * 60)

print()

print("Total Parameters")

print(f"{count_parameters(moe_model):,}")

print()

optimizer = torch.optim.AdamW(
    moe_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("=" * 60)
print("START TRAINING MoE")
print("=" * 60)

moe_train_loss = []
moe_val_loss = []

for step in range(MAX_ITERS):

    if step % EVAL_INTERVAL == 0:

        moe_model.eval()

        losses = {}

        with torch.no_grad():

            for split in ["train", "val"]:

                total = 0.0

                for _ in range(EVAL_ITERS):

                    x, y = get_batch(split)

                    _, loss = moe_model(x, y)

                    total += loss.item()

                losses[split] = total / EVAL_ITERS

        moe_model.train()

        train_loss = losses["train"]
        val_loss = losses["val"]

        moe_train_loss.append(train_loss)
        moe_val_loss.append(val_loss)

        print(
            f"Step {step:5d} | "
            f"Train {train_loss:.4f} | "
            f"Val {val_loss:.4f}"
        )

    x, y = get_batch("train")

    _, loss = moe_model(x, y)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

print()

print("=" * 60)
print("MoE TRAINING FINISHED")
print("=" * 60)

TinyGPT-MoE

Total Parameters
2,399,313

START TRAINING MoE
Step     0 | Train 4.3715 | Val 4.3701
Step   500 | Train 2.2614 | Val 2.2781
Step  1000 | Train 1.9857 | Val 2.0573
Step  1500 | Train 1.8306 | Val 1.9322
Step  2000 | Train 1.7185 | Val 1.8712
Step  2500 | Train 1.6526 | Val 1.8246
Step  3000 | Train 1.5941 | Val 1.7771
Step  3500 | Train 1.5550 | Val 1.7348
Step  4000 | Train 1.5221 | Val 1.6905
Step  4500 | Train 1.4935 | Val 1.6825

MoE TRAINING FINISHED


**Text Generation**

In [ ]:
# ==========================================================
# Generate Text (MoE GPT)
# ==========================================================

context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=DEVICE
)

generated = moe_model.generate(

    context,

    max_new_tokens=500,

    temperature=0.8,

    top_k=20

)

generated_text = decode(
    generated[0].tolist()
)

print("=" * 60)
print("GENERATED TEXT (MoE GPT)")
print("=" * 60)

print()

print(generated_text)

GENERATED TEXT (MoE GPT)



MARCIUS:
I pleasu you, be present composions it lost
him, tell peace to be against like.

DUCHESS OF YORK:
Your holy but and prince of the gift you flatter
With the trong of my lady with his such of greatly by pain.

KING HENRY VI:
How not, course love mighty to do prove, he bear
The duke early. God so toward and remour
With hearts softer, for their friends, and or structions wereign,
To distructor; I shall shall so her to hours faths.

CAPULET:
My thank get proud in them the will.

AUTOLYCUS:



**A comparison**

In [ ]:
# ==========================================================
# Block 14C : Dense GPT vs MoE GPT Comparison
# ==========================================================

import time

print("=" * 80)
print("DENSE GPT vs MoE GPT")
print("=" * 80)

# ----------------------------------------------------------
# Parameter Count
# ----------------------------------------------------------

dense_params = count_parameters(model)
moe_params = count_parameters(moe_model)

# ----------------------------------------------------------
# Final Losses
# ----------------------------------------------------------

dense_train_loss = loss_history[-1]
dense_val_loss = val_history[-1]

moe_train_loss_final = moe_train_loss[-1]
moe_val_loss_final = moe_val_loss[-1]

# ----------------------------------------------------------
# Generation Context
# ----------------------------------------------------------

context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=DEVICE
)

# ==========================================================
# Dense GPT Generation
# ==========================================================

if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.time()

dense_output = model.generate(
    context.clone(),
    max_new_tokens=200,
    temperature=0.8,
    top_k=20
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

dense_time = time.time() - start

# ==========================================================
# MoE GPT Generation
# ==========================================================

if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.time()

moe_output = moe_model.generate(
    context.clone(),
    max_new_tokens=200,
    temperature=0.8,
    top_k=20
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

moe_time = time.time() - start

# ==========================================================
# Decode Outputs
# ==========================================================

dense_text = decode(dense_output[0].tolist())

moe_text = decode(moe_output[0].tolist())

# ==========================================================
# Print Comparison
# ==========================================================

print()

print(f"{'Metric':<35}{'Dense GPT':<20}{'MoE GPT':<20}")
print("-" * 75)

print(f"{'Total Parameters':<35}{dense_params:<20,}{moe_params:<20,}")

print(f"{'Training Loss':<35}{dense_train_loss:<20.4f}{moe_train_loss_final:<20.4f}")

print(f"{'Validation Loss':<35}{dense_val_loss:<20.4f}{moe_val_loss_final:<20.4f}")

print(f"{'Generation Time (seconds)':<35}{dense_time:<20.4f}{moe_time:<20.4f}")

print()

print("=" * 80)
print("GENERATED TEXT : DENSE GPT")
print("=" * 80)
print()

print(dense_text)

print()

print("=" * 80)
print("GENERATED TEXT : MoE GPT")
print("=" * 80)
print()

print(moe_text)

# ==========================================================
# Analysis
# ==========================================================

print()

print("=" * 80)
print("COMPARISON SUMMARY")
print("=" * 80)

print()

if dense_train_loss < moe_train_loss_final:
    print("✓ Dense GPT achieved lower training loss.")
else:
    print("✓ MoE GPT achieved lower training loss.")

if dense_val_loss < moe_val_loss_final:
    print("✓ Dense GPT achieved lower validation loss.")
else:
    print("✓ MoE GPT achieved lower validation loss.")

if dense_time < moe_time:
    print("✓ Dense GPT generated text faster.")
else:
    print("✓ MoE GPT generated text faster.")

print()

print(f"Dense GPT Parameters : {dense_params:,}")
print(f"MoE GPT Parameters   : {moe_params:,}")

print()

print("Note:")
print("- This educational MoE computes every expert on the full batch before masking.")
print("- Therefore, it is expected to be slower than Dense GPT.")
print("- Production MoE implementations dispatch only the selected tokens to each expert,")
print("  which is where the computational efficiency comes from.")

print("=" * 80)

DENSE GPT vs MoE GPT

Metric                             Dense GPT           MoE GPT             
---------------------------------------------------------------------------
Total Parameters                   816,705             2,399,313           
Training Loss                      1.5782              1.4935              
Validation Loss                    1.7494              1.6825              
Generation Time (seconds)          2.0451              3.0249              

GENERATED TEXT : DENSE GPT


Fairst no mot, why dectites a crown will his likes.

Second JULIET:
I would be may you, to good this very at is him day.

MERCUTIO:
O lord, I'll not counsage you make a stant denies
Is that strengt's 

GENERATED TEXT : MoE GPT



WARWICK:
Well more thy poor belies thee hath be any lead
With convented and at their his cottense,
Then he will his betty be sit thun the course.

SICINIUS:

Peconce and let him,
And uncle, the his h

COMPARISON SUMMARY

✓ MoE GPT achieved lower training loss.
✓

# Simulations

In [ ]:
# app.py
import gradio as gr
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)

EMBED_DIM = 8
NUM_EXPERTS = 4
TOP_K = 2

class Expert(nn.Module):
    def __init__(self, idx):
        super().__init__()
        self.idx = idx
        self.fc1 = nn.Linear(EMBED_DIM, 16)
        self.fc2 = nn.Linear(16, EMBED_DIM)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

class Router(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(EMBED_DIM, NUM_EXPERTS)

    def forward(self, x):
        return F.softmax(self.fc(x), dim=-1)

router = Router()
experts = [Expert(i+1) for i in range(NUM_EXPERTS)]

def text_embedding(text):
    vec = torch.zeros(EMBED_DIM)
    if not text.strip():
        return vec
    b = text.encode("utf-8")
    for i, v in enumerate(b):
        vec[i % EMBED_DIM] += v / 255.0
    return vec

def run_moe(text):
    x = text_embedding(text)
    probs = router(x)
    values, indices = torch.topk(probs, TOP_K)

    outputs = []
    final = torch.zeros(EMBED_DIM)

    explanation = []
    explanation.append("## Router Probabilities\n")

    for i, p in enumerate(probs):
        explanation.append(f"Expert {i+1}: **{p.item():.3f}**")

    explanation.append("\n### Selected Experts")
    for idx, w in zip(indices, values):
        explanation.append(f"- Expert {idx.item()+1} (weight={w.item():.3f})")

    explanation.append("\n### Expert Outputs")

    for idx, w in zip(indices, values):
        out = experts[idx.item()](x)
        outputs.append(
            f"Expert {idx.item()+1}\n"
            f"Weight: {w.item():.3f}\n"
            f"Output:\n{out.detach().numpy()}"
        )
        final += w * out

    fig, ax = plt.subplots(figsize=(5,3))
    ax.bar(
        [f"E{i+1}" for i in range(NUM_EXPERTS)],
        probs.detach().numpy()
    )
    ax.set_ylim(0,1)
    ax.set_ylabel("Probability")
    ax.set_title("Router Decision")
    plt.tight_layout()

    return (
        "\n".join(explanation),
        "\n\n".join(outputs),
        str(final.detach().numpy()),
        fig
    )

with gr.Blocks(title="Simple Mixture of Experts Demo") as demo:
    gr.Markdown("# 🧠 Mixture of Experts (MoE) Visualization")
    gr.Markdown(
        "Type a sentence. The router computes probabilities, "
        "selects the Top-2 experts, and combines only their outputs."
    )

    inp = gr.Textbox(
        label="Input Text",
        value="The cat is sleeping."
    )

    run = gr.Button("Run MoE")

    explanation = gr.Markdown()
    expert_box = gr.Textbox(
        label="Selected Expert Outputs",
        lines=14
    )

    final_output = gr.Textbox(
        label="Final MoE Output Vector",
        lines=4
    )

    chart = gr.Plot()

    run.click(
        run_moe,
        inputs=inp,
        outputs=[
            explanation,
            expert_box,
            final_output,
            chart
        ]
    )

demo.launch()



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ced915c74d5bac2046.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

import time
import random
import numpy as np
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
model.eval()

EXPERTS = [
    "🧮 Math",
    "💻 Coding",
    "🔬 Science",
    "📖 History",
    "🌍 Geography",
    "🩺 Medical",
    "🎨 Creative",
    "💬 General"
]

KEYWORDS = {
    0:["math","matrix","algebra","equation"],
    1:["python","code","java","bug","program"],
    2:["science","physics","biology","chemistry"],
    3:["history","war","empire"],
    4:["country","capital","map"],
    5:["doctor","medicine","health","disease"],
    6:["story","poem","creative"],
    7:["what","why","who","how"]
}

def router(prompt):
    scores=np.random.rand(8)*0.2
    p=prompt.lower()
    for i,words in KEYWORDS.items():
        for w in words:
            if w in p:
                scores[i]+=1.2
    probs=np.exp(scores)
    probs/=probs.sum()
    idx=np.argsort(probs)[::-1][:2]
    return probs,idx

def chat(prompt,temp,max_tokens):
    probs,top2=router(prompt)
    messages=[{"role":"user","content":prompt}]
    text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=tokenizer(text,return_tensors="pt").to(model.device)
    t0=time.time()
    with torch.no_grad():
        out=model.generate(
            **inputs,
            max_new_tokens=int(max_tokens),
            do_sample=True,
            temperature=float(temp),
            top_p=0.9
        )
    dt=time.time()-t0
    ans=tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True)
    table=[]
    for i,e in enumerate(EXPERTS):
        table.append([e,round(float(probs[i]),3),"✅" if i in top2 else ""])
    info=f"""Inference: {dt:.2f} s
Generated Tokens: {out.shape[1]-inputs["input_ids"].shape[1]}
Tokens/sec: {(out.shape[1]-inputs["input_ids"].shape[1])/max(dt,1e-6):.2f}

Active Experts:
1. {EXPERTS[top2[0]]}
2. {EXPERTS[top2[1]]}
"""
    return ans,table,info

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 SmolLM2 + MoE Visualizer")
    with gr.Row():
        with gr.Column():
            prompt=gr.Textbox(lines=6,label="Prompt")
            temp=gr.Slider(0.1,1.5,0.7,step=0.1,label="Temperature")
            mx=gr.Slider(32,256,128,step=32,label="Max Tokens")
            btn=gr.Button("Generate")
        with gr.Column():
            out=gr.Textbox(lines=14,label="LLM Response")
            stats=gr.Textbox(lines=6,label="Performance")
    table=gr.Dataframe(headers=["Expert","Probability","Selected"],interactive=False)
    btn.click(chat,[prompt,temp,mx],[out,table,stats])

demo.launch()



Loading model...


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

/tmp/ipykernel_16860/185625112.py:82: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://63c0d49dd02ce14c3d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
